# 🦜 The Agent Zoo — Build 5 Agent Architectures (Self-Graded)

You will build **five** different agent architectures with **LangGraph + Groq**, run each one for real, and finish by deciding **which agent wins which scenario**.

| You build | The big idea |
|---|---|
| 🔁 **ReAct** | Think → act → observe, in a loop |
| 🗂️ **Plan-and-Execute** | Plan all steps first, then execute |
| 🪞 **Reflexion** | Fail, reflect in words, retry smarter |
| 💻 **CodeAct** | Write & run Python as the action |
| 🌳 **Tree of Thoughts** | Branch, score, keep the best path |

**Your job** for each agent: write the **prompt**, write a small **logic helper**, and **wire the graph**. The LLM-calling node functions are given to you — the *thinking* parts are yours.

| Rank | Score |
|---|---|
| 🌱 Prompt Apprentice | 0–25% |
| 🔁 Loop Wrangler | 25–45% |
| 🗂️ Planner | 45–62% |
| 🪞 Reflective Builder | 62–78% |
| 🌳 Branch Explorer | 78–92% |
| 🏛️ Agent Architect | 92–100% |

**17 tasks · 120 points.** Hints hide in **💡 dropdowns**. Run each **✅ CHECK** to score yourself.


## ⚙️ Setup 1 — Install

In [ ]:
!pip install -q langchain-groq langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.2 MB/s eta 0:00:00


## ⚙️ Setup 2 — API key + model
🔑 Put your Groq key in Colab **Secrets** (🔑 icon, left sidebar) named **`groq_api`** — same as the lecture.

In [ ]:
from google.colab import userdata
from langchain_groq import ChatGroq

key = userdata.get('groq_api') or "MISSING_KEY"
llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0, api_key=key)

print("LLM ready." if key != "MISSING_KEY" else "WARNING: add your 'groq_api' secret, then re-run.")


LLM ready.


## ⚙️ Setup 3 — Common imports + grader *(just run it)*

In [ ]:
from typing import Annotated, TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
import io, contextlib, re

def has_all(text, words):
    t = (text or "").lower()
    return all(w in t for w in words)
def has_any(text, words):
    t = (text or "").lower()
    return any(w in t for w in words)

class Grader:
    RANKS=[(0.00,"🌱 Prompt Apprentice","Words are your first tool."),
           (0.25,"🔁 Loop Wrangler","Your agents loop without spinning out."),
           (0.45,"🗂️ Planner","You plan before you leap."),
           (0.62,"🪞 Reflective Builder","Your agents learn from mistakes."),
           (0.78,"🌳 Branch Explorer","You search the space of ideas."),
           (0.92,"🏛️ Agent Architect","You pick the right agent for the job.")]
    def __init__(self,total=120): self.scores={}; self.max_pts={}; self.total=total
    @property
    def earned(self): return sum(self.scores.values())
    def _bar(self,p,w=30): f=int(round(p*w)); return "#"*f+"-"*(w-f)
    def _rank(self):
        p=self.earned/self.total; r=self.RANKS[0]
        for t,n,b in self.RANKS:
            if p>=t: r=(t,n,b)
        return r
    def check(self,tid,passed,pts,ok_msg="",fail_msg=""):
        self.max_pts[tid]=pts; already=self.scores.get(tid,0)==pts
        if passed:
            self.scores[tid]=pts
            print(("Already complete" if already else "PASS  +"+str(pts)+" pts")+"  [Task "+tid+"]")
            if ok_msg and not already: print("   "+ok_msg)
        else:
            if not already: self.scores[tid]=0
            print("Not passing yet  (0 / "+str(pts)+" pts)  [Task "+tid+"]")
            if fail_msg: print("   "+fail_msg)
        p=self.earned/self.total; _,n,b=self._rank()
        print()
        print("   "+str(self.earned)+"/"+str(self.total)+"  ["+self._bar(p)+"]  "+str(round(p*100))+"%")
        print("   Rank: "+n+" - "+b); print()
    def scorecard(self):
        p=self.earned/self.total; _,n,b=self._rank()
        print("="*54); print("  FINAL SCORECARD"); print("="*54)
        for tid in sorted(self.max_pts):
            e,t=self.scores.get(tid,0),self.max_pts[tid]
            mark="[x]" if e==t else ("[~]" if e>0 else "[ ]")
            print("  "+mark+"  Task "+tid.ljust(5)+" "+str(e).rjust(3)+"/"+str(t))
        print("  "+"-"*44)
        print("  TOTAL  "+str(self.earned).rjust(3)+"/"+str(self.total)+"   ["+self._bar(p)+"]  "+str(round(p*100))+"%")
        print("  Rank: "+n); print("  "+b); print("="*54)

grader=Grader(120)
print("Grader ready. 120 points across 17 tasks.")


Grader ready. 120 points across 17 tasks.


---
# 🔁 Part 1 · ReAct

**The loop:** the model writes a **Thought** (what should I do?), takes an **Action** (calls a tool), sees an **Observation** (the result), and repeats until it can answer.

The Thought step does real work — it's not narration. Stating "I need the population before I can divide" literally steers the next action. But ReAct has a famous failure mode: **repetition loops** — it calls the same tool with the same input over and over because it never notices the tool already answered. We'll guard against that.


## TASK R1 — The ReAct system prompt &nbsp;`6 pts`

Write the system prompt (store in **`react_system_prompt`**) that turns the model into a ReAct agent. It should push the model to **reason about its approach before each step** and to **use the available tools** instead of guessing.


In [ ]:
# TASK R1 — set react_system_prompt = "..."
react_system_prompt = """Your task is to do reasoning for next steps,
then use appropriate tools and stop when got final answer."""

<details><summary>💡 Hint</summary>

Fill in the blanks of something like:
```python
react_system_prompt = (
    "You are a careful assistant. Solve the task step by step. "
    "Before each action, briefly REASON about what you need next. "
    "Use the available TOOLS to get facts instead of guessing, and stop once you can answer."
)
```
</details>

In [ ]:
# ✅ CHECK R1
try:
    ok = has_any(react_system_prompt, ["reason","think","thought","step"]) and has_any(react_system_prompt, ["tool"]) and len(react_system_prompt) > 60
    grader.check("R1", ok, 6, ok_msg="Prompt encourages reasoning + tool use.",
        fail_msg="Mention reasoning/thinking step-by-step AND using tools (and write a full sentence or two).")
except NameError:
    grader.check("R1", False, 6, fail_msg="`react_system_prompt` not defined.")


PASS  +6 pts  [Task R1]
   Prompt encourages reasoning + tool use.

   6/120  [##----------------------------]  5%
   Rank: 🌱 Prompt Apprentice - Words are your first tool.



## TASK R2 — Loop detector &nbsp;`8 pts`

Write **`detect_loop(history)`** where `history` is a list of strings, each a signature of a past tool call (e.g. `"search:France"`). Return **`True`** only if the **last two** signatures are identical (the agent is stuck repeating), otherwise **`False`**. Fewer than 2 entries → `False`.


In [ ]:
# TASK R2 — def detect_loop(history): ...
def detect_loop(history):
  return len(history) >= 2 and history[-1] == history[-2]

<details><summary>💡 Hint</summary>

```python
def detect_loop(history):
    return len(history) >= 2 and history[-1] == history[-2]
```
</details>

In [ ]:
# ✅ CHECK R2
try:
    ok = (detect_loop(["search:France","search:France"]) is True
          and detect_loop(["search:France","calc:2+2"]) is False
          and detect_loop(["only one"]) is False
          and detect_loop([]) is False)
    grader.check("R2", ok, 8, ok_msg="Loop detection works.",
        fail_msg="Return True only when the last two entries match; False for <2 entries.")
except NameError:
    grader.check("R2", False, 8, fail_msg="`detect_loop` not defined.")
except Exception as e:
    grader.check("R2", False, 8, fail_msg="Error: "+str(e))


PASS  +8 pts  [Task R2]
   Loop detection works.

   14/120  [####--------------------------]  12%
   Rank: 🌱 Prompt Apprentice - Words are your first tool.



### Given — tools & nodes for ReAct *(run this)*

In [ ]:
def multiply(a: int, b: int) -> int:
    """Multiply two numbers together."""
    return a * b

def word_count(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())

react_tools = [multiply, word_count]
llm_with_tools = llm.bind_tools(react_tools)

class ReActState(TypedDict):
    messages: Annotated[list, add_messages]

def reason_node(state):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

def react_route(state):
    last = state["messages"][-1]
    return "tools" if last.tool_calls else END

react_tool_node = ToolNode(react_tools)
print("ReAct nodes ready: reason_node, react_route, react_tool_node")


ReAct nodes ready: reason_node, react_route, react_tool_node


## TASK R3 — Wire the ReAct loop &nbsp;`6 pts`

Build the graph and compile it into **`react_agent`**. You need the loop:
`START → reason`, then a conditional from `reason` (to **tools** or **END**), and `tools → reason`.

Use the given `reason_node`, `react_route`, `react_tool_node`. Name the nodes **`"reason"`** and **`"tools"`**.


In [ ]:
# TASK R3 — build StateGraph(ReActState) and compile to react_agent
builder = StateGraph(ReActState)
builder.add_node("reason", reason_node)
builder.add_node("tools", react_tool_node)
builder.add_edge(START, "reason")
builder.add_conditional_edges("reason", react_route)
builder.add_edge("tools", "reason")
react_agent = builder.compile()

<details><summary>💡 Hint</summary>

```python
b = StateGraph(ReActState)
b.add_node("reason", reason_node)
b.add_node("tools", react_tool_node)
b.add_edge(START, "reason")
b.add_conditional_edges("reason", react_route)
b.add_edge("tools", "reason")
react_agent = b.compile()
```
</details>

In [ ]:
# ✅ CHECK R3
try:
    nodes = set(react_agent.get_graph().nodes.keys())
    ok = {"reason","tools"}.issubset(nodes)
    grader.check("R3", ok, 6, ok_msg="ReAct loop wired with nodes "+str(sorted(n for n in nodes if not n.startswith("__")))+".",
        fail_msg="Nodes found: "+str(sorted(nodes))+" (need 'reason' and 'tools').")
except NameError:
    grader.check("R3", False, 6, fail_msg="`react_agent` not defined.")
except Exception as e:
    grader.check("R3", False, 6, fail_msg="Error: "+str(e))


PASS  +6 pts  [Task R3]
   ReAct loop wired with nodes ['reason', 'tools'].

   20/120  [#####-------------------------]  17%
   Rank: 🌱 Prompt Apprentice - Words are your first tool.



### ▶️ Run your ReAct agent live *(optional)*

In [ ]:
try:
    q = "What is 23 times 19, and how many words are in the sentence 'the quick brown fox jumps'?"
    out = react_agent.invoke({"messages": [("system", react_system_prompt), ("user", q)]})
    print(out["messages"][-1].content)
except Exception as e:
    print("Live run needs your API key. Error:", e)


The result of 23 times 19 is 437, and there are 5 words in the sentence 'the quick brown fox jumps'.


---
# 🗂️ Part 2 · Plan-and-Execute

ReAct decides step 2 only after seeing step 1. **Plan-and-Execute** decides **all** the steps **first**, then a separate executor works through them one by one. With full attention on the whole task, the planner writes better-structured plans and avoids redundant work — great for long, predictable tasks (research reports, pipelines with known stages).


## TASK P1 — The planner prompt &nbsp;`6 pts`

Write **`planner_prompt`**: it should make the model break a task into a short, **numbered list of concrete steps** — and nothing else.


In [ ]:
# TASK P1 — set planner_prompt = "..."
planner_prompt = """You are a planner. Break the user's task into 3 to 5 concrete steps.
Respond ONLY as a numbered list, one step per line, like '1. ...'. No extra text."""

<details><summary>💡 Hint</summary>

```python
planner_prompt = (
    "You are a planner. Break the user's task into 3 to 5 concrete steps. "
    "Respond ONLY as a numbered list, one step per line, like '1. ...'. No extra text."
)
```
</details>

In [ ]:
# ✅ CHECK P1
try:
    ok = has_any(planner_prompt, ["step"]) and has_any(planner_prompt, ["plan","list","number","numbered"]) and len(planner_prompt) > 60
    grader.check("P1", ok, 6, ok_msg="Planner prompt asks for a numbered list of steps.",
        fail_msg="Ask the model for a numbered list / plan of steps (full sentence).")
except NameError:
    grader.check("P1", False, 6, fail_msg="`planner_prompt` not defined.")


Already complete  [Task P1]

   40/120  [##########--------------------]  33%
   Rank: 🔁 Loop Wrangler - Your agents loop without spinning out.



## TASK P2 — Parse the plan &nbsp;`8 pts`

The planner returns text like:
```
1. Research the topic
2. Write a draft
3. Review and finalize
```
Write **`parse_plan(text)`** that turns this into a **list of step strings** — one per numbered line. (For the sample above it returns 3 items.)


In [ ]:
# TASK P2 — def parse_plan(text): ...
def parse_plan(text):
    steps = []
    for line in text.splitlines():
        s = line.strip()
        if s and s[0].isdigit():
            steps.append(s.split(".", 1)[1].strip() if "." in s else s)
    return steps

<details><summary>💡 Hint</summary>

Walk the lines, keep ones that start with a digit, and drop the leading "N.":
```python
def parse_plan(text):
    steps = []
    for line in text.splitlines():
        s = line.strip()
        if s and s[0].isdigit():
            steps.append(s.split(".", 1)[1].strip() if "." in s else s)
    return steps
```
</details>

In [ ]:
# ✅ CHECK P2
try:
    sample = "1. Research the topic\n2. Write a draft\n3. Review and finalize"
    out = parse_plan(sample)
    ok = isinstance(out, list) and len(out) == 3 and all(isinstance(s, str) and len(s) > 0 for s in out)
    sample4 = "1. a\n2. b\n3. c\n4. d"
    ok = ok and len(parse_plan(sample4)) == 4
    grader.check("P2", ok, 8, ok_msg="Parsed "+str(len(out))+" steps from the sample plan.",
        fail_msg="Should return one string per numbered line. Got: "+str(out)[:120])
except NameError:
    grader.check("P2", False, 8, fail_msg="`parse_plan` not defined.")
except Exception as e:
    grader.check("P2", False, 8, fail_msg="Error: "+str(e))


PASS  +8 pts  [Task P2]
   Parsed 3 steps from the sample plan.

   34/120  [########----------------------]  28%
   Rank: 🔁 Loop Wrangler - Your agents loop without spinning out.



### Given — nodes for Plan-and-Execute *(run this)*

In [ ]:
class PlanState(TypedDict):
    task: str
    plan: list
    step_index: int
    results: list

def planner_node(state):
    text = llm.invoke(planner_prompt + "\n\nTask: " + state["task"]).content
    return {"plan": parse_plan(text), "step_index": 0, "results": []}

def executor_node(state):
    i = state["step_index"]
    step = state["plan"][i]
    out = llm.invoke("Carry out ONLY this step and report the result briefly:\n" + step).content
    return {"results": state["results"] + [out], "step_index": i + 1}

def more_steps(state):
    return "executor" if state["step_index"] < len(state["plan"]) else END

print("Plan-Execute nodes ready: planner_node, executor_node, more_steps")


Plan-Execute nodes ready: planner_node, executor_node, more_steps


## TASK P3 — Wire Plan-and-Execute &nbsp;`6 pts`

Compile into **`plan_execute_agent`**. Flow: `START → planner → executor`, then a conditional from `executor` (back to **executor** while steps remain, else **END**). Node names: **`"planner"`**, **`"executor"`**.


In [ ]:
# TASK P3 — build and compile plan_execute_agent
builder = StateGraph(PlanState)
builder.add_node("planner", planner_node)
builder.add_node("executor", executor_node)
builder.add_edge(START, "planner")
builder.add_conditional_edges("executor", more_steps)
plan_execute_agent  = builder.compile()

<details><summary>💡 Hint</summary>

```python
b = StateGraph(PlanState)
b.add_node("planner", planner_node)
b.add_node("executor", executor_node)
b.add_edge(START, "planner")
b.add_edge("planner", "executor")
b.add_conditional_edges("executor", more_steps)
plan_execute_agent = b.compile()
```
</details>

In [ ]:
# ✅ CHECK P3
try:
    nodes = set(plan_execute_agent.get_graph().nodes.keys())
    ok = {"planner","executor"}.issubset(nodes)
    grader.check("P3", ok, 6, ok_msg="Plan-Execute wired with "+str(sorted(n for n in nodes if not n.startswith("__")))+".",
        fail_msg="Nodes found: "+str(sorted(nodes))+" (need 'planner' and 'executor').")
except NameError:
    grader.check("P3", False, 6, fail_msg="`plan_execute_agent` not defined.")
except Exception as e:
    grader.check("P3", False, 6, fail_msg="Error: "+str(e))


Already complete  [Task P3]

   40/120  [##########--------------------]  33%
   Rank: 🔁 Loop Wrangler - Your agents loop without spinning out.



### ▶️ Run Plan-and-Execute live *(optional)*

In [ ]:
try:
    res = plan_execute_agent.invoke({"task": "Plan a simple 3-step approach to learn a new language."})
    print("PLAN:", res["plan"])
    for i, r in enumerate(res["results"], 1):
        print(f"\n--- result of step {i} ---\n{r}")
except Exception as e:
    print("Live run needs your API key. Error:", e)


PLAN: ['Set specific and achievable language learning goals, such as basic conversation skills or reading comprehension, and choose the resources and materials needed to support these goals.', 'Create a study schedule and routine, including daily or weekly practice sessions, to ensure consistent progress and reinforcement of new language skills.', 'Immerse yourself in the language by listening to music, watching TV shows or movies, and engaging in conversations with native speakers, either in person or online, to practice speaking and listening skills.']


---
# 🪞 Part 3 · Reflexion

A plain retry repeats the same mistake. **Reflexion** adds a step: after failing, the agent writes a sentence of **verbal reflection** ("I failed because I searched the common name, not the technical term — next time I'll use the technical term"), stores it, and reads it on the next attempt. Three parts: an **Actor** that tries, an **Evaluator** that judges, and a **Reflector** that writes the lesson. Memory is capped at the few most recent reflections so the important ones don't get buried.


## TASK X1 — The reflection prompt &nbsp;`6 pts`

Write **`reflection_prompt`**: given the failed attempt, it should make the model explain **why it went wrong** and **what to do differently next time** — in one short paragraph.


In [ ]:
# TASK X1 — set reflection_prompt = "..."
reflection_prompt = """Your previous attempt FAILED.
In 1-2 sentences, explain exactly what went WRONG and what you will do DIFFERENTLY next time to IMPROVE.
Be specific."""

<details><summary>💡 Hint</summary>

```python
reflection_prompt = (
    "Your previous attempt FAILED. In 1-2 sentences, explain exactly what went WRONG "
    "and what you will do DIFFERENTLY next time to IMPROVE. Be specific."
)
```
</details>

In [ ]:
# ✅ CHECK X1
try:
    ok = has_any(reflection_prompt, ["fail","wrong","mistake"]) and has_any(reflection_prompt, ["improve","better","next time","differently","different"]) and len(reflection_prompt) > 60
    grader.check("X1", ok, 6, ok_msg="Reflection prompt asks what went wrong + how to improve.",
        fail_msg="Mention what went WRONG and what to do to IMPROVE next time.")
except NameError:
    grader.check("X1", False, 6, fail_msg="`reflection_prompt` not defined.")


PASS  +6 pts  [Task X1]
   Reflection prompt asks what went wrong + how to improve.

   46/120  [############------------------]  38%
   Rank: 🔁 Loop Wrangler - Your agents loop without spinning out.



## TASK X2 — Bounded reflection memory &nbsp;`8 pts`

Write **`add_reflection(memory, new_reflection, cap=3)`** that appends `new_reflection` to the `memory` list but keeps only the **most recent `cap`** items, and returns the updated list.


In [ ]:
# TASK X2 — def add_reflection(memory, new_reflection, cap=3): ...
def add_reflection(memory, new_reflection, cap=3):
    return (memory + [new_reflection])[-cap:]

<details><summary>💡 Hint</summary>

Slicing keeps the last `cap` items:
```python
def add_reflection(memory, new_reflection, cap=3):
    return (memory + [new_reflection])[-cap:]
```
</details>

In [ ]:
# ✅ CHECK X2
try:
    m = []
    for r in ["r1","r2","r3","r4","r5"]:
        m = add_reflection(m, r, cap=3)
    ok = (m == ["r3","r4","r5"]) and (add_reflection([], "first", 3) == ["first"])
    grader.check("X2", ok, 8, ok_msg="Memory keeps the 3 most recent: "+str(m),
        fail_msg="After adding r1..r5 with cap=3 you should keep ['r3','r4','r5']. Got "+str(m))
except NameError:
    grader.check("X2", False, 8, fail_msg="`add_reflection` not defined.")
except Exception as e:
    grader.check("X2", False, 8, fail_msg="Error: "+str(e))


PASS  +8 pts  [Task X2]
   Memory keeps the 3 most recent: ['r3', 'r4', 'r5']

   54/120  [##############----------------]  45%
   Rank: 🗂️ Planner - You plan before you leap.



### Given — nodes for Reflexion *(run this)*

The **Evaluator** here is a simple deterministic rule: the task is *"Reply with a sentence that SHOUTS the word BANANA in capitals,"* and success = the answer contains `BANANA`. This makes the loop terminate cleanly so you can watch reflection work.

In [ ]:
class ReflexionState(TypedDict):
    task: str
    answer: str
    reflections: list
    attempts: int
    success: bool

def actor_node(state):
    past = "\n".join(state.get("reflections", []))
    prompt = ("Task: " + state["task"] + "\n"
              + ("Lessons from past attempts:\n" + past + "\n" if past else "")
              + "Now give your best attempt.")
    ans = llm.invoke(prompt).content
    return {"answer": ans, "attempts": state.get("attempts", 0) + 1}

def evaluate_node(state):
    return {"success": "BANANA" in state["answer"]}

def reflexion_route(state):
    return END if (state["success"] or state["attempts"] >= 3) else "reflect"

def reflect_node(state):
    r = llm.invoke(reflection_prompt + "\n\nMy failed answer was: " + state["answer"]).content
    return {"reflections": add_reflection(state.get("reflections", []), r, cap=3)}

print("Reflexion nodes ready: actor_node, evaluate_node, reflexion_route, reflect_node")


Reflexion nodes ready: actor_node, evaluate_node, reflexion_route, reflect_node


## TASK X3 — Wire the Reflexion loop &nbsp;`6 pts`

Compile into **`reflexion_agent`**. Flow: `START → actor → evaluate`, then a conditional from `evaluate` (to **reflect** or **END**), and `reflect → actor`. Node names: **`"actor"`**, **`"evaluate"`**, **`"reflect"`**.


In [ ]:
# TASK X3 — build and compile reflexion_agent
b = StateGraph(ReflexionState)
b.add_node("actor", actor_node)
b.add_node("evaluate", evaluate_node)
b.add_node("reflect", reflect_node)
b.add_edge(START, "actor")
b.add_edge("actor", "evaluate")
b.add_conditional_edges("evaluate", reflexion_route)
b.add_edge("reflect", "actor")
reflexion_agent = b.compile()

<details><summary>💡 Hint</summary>

```python
b = StateGraph(ReflexionState)
b.add_node("actor", actor_node)
b.add_node("evaluate", evaluate_node)
b.add_node("reflect", reflect_node)
b.add_edge(START, "actor")
b.add_edge("actor", "evaluate")
b.add_conditional_edges("evaluate", reflexion_route)
b.add_edge("reflect", "actor")
reflexion_agent = b.compile()
```
</details>

In [ ]:
# ✅ CHECK X3
try:
    nodes = set(reflexion_agent.get_graph().nodes.keys())
    ok = {"actor","evaluate","reflect"}.issubset(nodes)
    grader.check("X3", ok, 6, ok_msg="Reflexion loop wired with "+str(sorted(n for n in nodes if not n.startswith("__")))+".",
        fail_msg="Nodes found: "+str(sorted(nodes))+" (need 'actor','evaluate','reflect').")
except NameError:
    grader.check("X3", False, 6, fail_msg="`reflexion_agent` not defined.")
except Exception as e:
    grader.check("X3", False, 6, fail_msg="Error: "+str(e))


PASS  +6 pts  [Task X3]
   Reflexion loop wired with ['actor', 'evaluate', 'reflect'].

   60/120  [###############---------------]  50%
   Rank: 🗂️ Planner - You plan before you leap.



### ▶️ Run Reflexion live *(optional)*

In [ ]:
try:
    init = {"task": "Reply with a sentence that SHOUTS the word BANANA in capital letters.",
            "answer": "", "reflections": [], "attempts": 0, "success": False}
    res = reflexion_agent.invoke(init)
    print("Attempts:", res["attempts"], "| Success:", res["success"])
    print("Final answer:", res["answer"])
    print("Reflections collected:", len(res["reflections"]))
except Exception as e:
    print("Live run needs your API key. Error:", e)


Attempts: 1 | Success: True
Final answer: I just can't wait to eat a delicious BANANA for breakfast today.
Reflections collected: 0


---
# 💻 Part 4 · CodeAct

Most agents have a fixed toolbox. **CodeAct** gives the agent one *unbounded* tool: a Python interpreter. The action **is** code. Need to parse a date, sum a column, call an API? Just write the code. Code is composable, exact at arithmetic, and verifiable.

⚠️ **Safety:** running model-written code needs a **sandbox** in production. Here we run it inside your own Colab runtime for learning — never point this at anything important.


## TASK C1 — The CodeAct prompt &nbsp;`6 pts`

Write **`codeact_prompt`**: it should tell the model to solve the task by writing **Python code** that **prints** the answer, wrapped in a ```` ```python ```` code block.


In [ ]:
# TASK C1 — set codeact_prompt = "..."
codeact_prompt = """Solve the task by writing PYTHON code that PRINTS the final answer.
Return ONLY a python code block, like:\n```python\n# code here\nprint(answer)\n```"""

<details><summary>💡 Hint</summary>

```python
codeact_prompt = (
    "Solve the task by writing PYTHON code that PRINTS the final answer. "
    "Return ONLY a python code block, like:\n```python\n# code here\nprint(answer)\n```"
)
```
</details>

In [ ]:
# ✅ CHECK C1
try:
    ok = has_any(codeact_prompt, ["python","code"]) and has_any(codeact_prompt, ["print"]) and len(codeact_prompt) > 60
    grader.check("C1", ok, 6, ok_msg="CodeAct prompt asks for Python that prints the answer.",
        fail_msg="Tell the model to write PYTHON code that PRINTs the answer.")
except NameError:
    grader.check("C1", False, 6, fail_msg="`codeact_prompt` not defined.")


PASS  +6 pts  [Task C1]
   CodeAct prompt asks for Python that prints the answer.

   66/120  [################--------------]  55%
   Rank: 🗂️ Planner - You plan before you leap.



## TASK C2 — Extract & run code &nbsp;`8 pts`

Write **two** helpers:
1. **`extract_code(text)`** — pull the code out of a ```` ```python ... ``` ```` block. If there's no block, return the text as-is (stripped).
2. **`run_code(code)`** — execute the code and **return whatever it printed**, as a stripped string.


In [ ]:
# TASK C2 — def extract_code(text): ... and def run_code(code): ...
def extract_code(text):
    if "```" in text:
        block = text.split("```")[1]
        if block.startswith("python"):
            block = block[len("python"):]
        return block.strip()
    return text.strip()

def run_code(code):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        exec(code, {})
    return buf.getvalue().strip()

<details><summary>💡 Hint</summary>

```python
def extract_code(text):
    if "```" in text:
        block = text.split("```")[1]
        if block.startswith("python"):
            block = block[len("python"):]
        return block.strip()
    return text.strip()

def run_code(code):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        exec(code, {})
    return buf.getvalue().strip()
```
</details>

In [ ]:
# ✅ CHECK C2
try:
    ec = extract_code("Sure:\n```python\nprint(2 + 2)\n```\nDone")
    rc = run_code("print(7 * 6)")
    ok = (ec == "print(2 + 2)") and (rc == "42")
    grader.check("C2", ok, 8, ok_msg="extract_code and run_code both work (got '"+rc+"' from print(7*6)).",
        fail_msg="extract_code -> "+repr(ec)+" ; run_code('print(7*6)') -> "+repr(rc))
except NameError as e:
    grader.check("C2", False, 8, fail_msg="Missing: "+str(e))
except Exception as e:
    grader.check("C2", False, 8, fail_msg="Error: "+str(e))


PASS  +8 pts  [Task C2]
   extract_code and run_code both work (got '42' from print(7*6)).

   74/120  [##################------------]  62%
   Rank: 🗂️ Planner - You plan before you leap.



### Given — nodes for CodeAct *(run this)*

In [ ]:
class CodeActState(TypedDict):
    task: str
    code: str
    output: str
    attempts: int
    done: bool

def codegen_node(state):
    feedback = ("\nThe previous code printed: " + state["output"]) if state.get("output") else ""
    text = llm.invoke(codeact_prompt + "\n\nTask: " + state["task"] + feedback).content
    return {"code": extract_code(text), "attempts": state.get("attempts", 0) + 1}

def execute_node(state):
    try:
        out = run_code(state["code"])
        done = (out != "")
    except Exception as e:
        out, done = "Error: " + str(e), False
    return {"output": out, "done": done}

def codeact_route(state):
    return END if (state["done"] or state["attempts"] >= 2) else "codegen"

print("CodeAct nodes ready: codegen_node, execute_node, codeact_route")


CodeAct nodes ready: codegen_node, execute_node, codeact_route


## TASK C3 — Wire CodeAct &nbsp;`6 pts`

Compile into **`codeact_agent`**. Flow: `START → codegen → execute`, then a conditional from `execute` (back to **codegen** if it failed and attempts remain, else **END**). Node names: **`"codegen"`**, **`"execute"`**.


In [ ]:
# TASK C3 — build and compile codeact_agent
b = StateGraph(CodeActState)
b.add_node("codegen", codegen_node)
b.add_node("execute", execute_node)
b.add_edge(START, "codegen")
b.add_edge("codegen", "execute")
b.add_conditional_edges("execute", codeact_route)
codeact_agent = b.compile()

<details><summary>💡 Hint</summary>

```python
b = StateGraph(CodeActState)
b.add_node("codegen", codegen_node)
b.add_node("execute", execute_node)
b.add_edge(START, "codegen")
b.add_edge("codegen", "execute")
b.add_conditional_edges("execute", codeact_route)
codeact_agent = b.compile()
```
</details>

In [ ]:
# ✅ CHECK C3
try:
    nodes = set(codeact_agent.get_graph().nodes.keys())
    ok = {"codegen","execute"}.issubset(nodes)
    grader.check("C3", ok, 6, ok_msg="CodeAct wired with "+str(sorted(n for n in nodes if not n.startswith("__")))+".",
        fail_msg="Nodes found: "+str(sorted(nodes))+" (need 'codegen' and 'execute').")
except NameError:
    grader.check("C3", False, 6, fail_msg="`codeact_agent` not defined.")
except Exception as e:
    grader.check("C3", False, 6, fail_msg="Error: "+str(e))


PASS  +6 pts  [Task C3]
   CodeAct wired with ['codegen', 'execute'].

   80/120  [####################----------]  67%
   Rank: 🪞 Reflective Builder - Your agents learn from mistakes.



### ▶️ Run CodeAct live *(optional)*

In [ ]:
try:
    res = codeact_agent.invoke({"task": "What is the sum of all even numbers from 1 to 100?",
                                "code": "", "output": "", "attempts": 0, "done": False})
    print("CODE THE AGENT WROTE:\n", res["code"])
    print("\nACTUAL OUTPUT:", res["output"])
except Exception as e:
    print("Live run needs your API key. Error:", e)


CODE THE AGENT WROTE:
 # Calculate the sum of all even numbers from 1 to 100
even_sum = sum(i for i in range(1, 101) if i % 2 == 0)
print(even_sum)

ACTUAL OUTPUT: 2550


---
# 🌳 Part 5 · Tree of Thoughts

Linear reasoning commits to one path; if step 2 was a wrong turn, steps 3–10 are wasted. **Tree of Thoughts** generates **several** candidate approaches, **scores** each one, and continues from the most promising — exploring the space instead of guessing once. It's powerful but **expensive** (every branch is an LLM call). We'll build a one-level version: **generate → evaluate → select**.


## TASK T1 — Two prompts &nbsp;`6 pts`

Write **`thought_gen_prompt`** (asks the model for **several different** candidate approaches to a problem) and **`thought_eval_prompt`** (asks the model to **rate one approach with a score from 1 to 10**).


In [ ]:
# TASK T1 — set thought_gen_prompt and thought_eval_prompt
thought_gen_prompt = """Suggest 3 DIFFERENT high-level approaches to the problem below.
One per line, no numbering, each a single short sentence."""

thought_eval_prompt = """Rate how promising this approach is for solving the problem, with a SCORE from 1 to 10.
Reply with just the number."""

<details><summary>💡 Hint</summary>

```python
thought_gen_prompt = (
    "Suggest 3 DIFFERENT high-level approaches to the problem below. "
    "One per line, no numbering, each a single short sentence."
)
thought_eval_prompt = (
    "Rate how promising this approach is for solving the problem, with a SCORE from 1 to 10. "
    "Reply with just the number."
)
```
</details>

In [ ]:
# ✅ CHECK T1
try:
    ok_gen = has_any(thought_gen_prompt, ["different","multiple","several","approach","option","idea"]) and len(thought_gen_prompt) > 40
    ok_eval = has_any(thought_eval_prompt, ["score","rate","rating"]) and ("10" in thought_eval_prompt) and len(thought_eval_prompt) > 40
    grader.check("T1", ok_gen and ok_eval, 6,
        ok_msg="Generation prompt branches; evaluation prompt scores 1-10.",
        fail_msg="gen_ok="+str(ok_gen)+", eval_ok="+str(ok_eval)+" (gen: ask for several approaches; eval: rate 1-10).")
except NameError as e:
    grader.check("T1", False, 6, fail_msg="Missing: "+str(e))


PASS  +6 pts  [Task T1]
   Generation prompt branches; evaluation prompt scores 1-10.

   86/120  [######################--------]  72%
   Rank: 🪞 Reflective Builder - Your agents learn from mistakes.



## TASK T2 — Score parser & best picker &nbsp;`8 pts`

Write:
1. **`parse_score(text)`** — return the **first integer** found in the evaluator's reply (e.g. `"Score: 8/10"` → `8`). If none, return `0`.
2. **`select_best(thoughts, scores)`** — return the **thought with the highest score**.


In [ ]:
# TASK T2 — def parse_score(text): ... and def select_best(thoughts, scores): ...
def parse_score(text):
    m = re.search(r"\d+", text)
    return int(m.group()) if m else 0

def select_best(thoughts, scores):
    return thoughts[scores.index(max(scores))]

<details><summary>💡 Hint</summary>

```python
def parse_score(text):
    m = re.search(r"\d+", text)
    return int(m.group()) if m else 0

def select_best(thoughts, scores):
    return thoughts[scores.index(max(scores))]
```
</details>

In [ ]:
# ✅ CHECK T2
try:
    ok = (parse_score("I rate this 8 out of 10") == 8
          and parse_score("Score: 7/10") == 7
          and parse_score("no number here") == 0
          and select_best(["A","B","C"], [3, 9, 5]) == "B")
    grader.check("T2", ok, 8, ok_msg="Score parsing and best-branch selection both work.",
        fail_msg="parse_score('Score: 7/10')="+str(parse_score('Score: 7/10') if 'parse_score' in dir() else '?')+
                 " ; select_best(['A','B','C'],[3,9,5]) should be 'B'.")
except NameError as e:
    grader.check("T2", False, 8, fail_msg="Missing: "+str(e))
except Exception as e:
    grader.check("T2", False, 8, fail_msg="Error: "+str(e))


PASS  +8 pts  [Task T2]
   Score parsing and best-branch selection both work.

   94/120  [########################------]  78%
   Rank: 🌳 Branch Explorer - You search the space of ideas.



### Given — nodes for Tree of Thoughts *(run this)*

In [ ]:
class ToTState(TypedDict):
    problem: str
    thoughts: list
    scores: list
    best: str
    answer: str

def generate_node(state):
    text = llm.invoke(thought_gen_prompt + "\n\nProblem: " + state["problem"]).content
    thoughts = [line.strip() for line in text.splitlines() if line.strip()][:3]
    return {"thoughts": thoughts}

def evaluate_branches_node(state):
    scores = []
    for t in state["thoughts"]:
        reply = llm.invoke(thought_eval_prompt + "\n\nProblem: " + state["problem"] + "\nApproach: " + t).content
        scores.append(parse_score(reply))
    return {"scores": scores}

def select_node(state):
    best = select_best(state["thoughts"], state["scores"])
    answer = llm.invoke("Solve the problem using this approach.\nProblem: " + state["problem"] + "\nApproach: " + best).content
    return {"best": best, "answer": answer}

print("ToT nodes ready: generate_node, evaluate_branches_node, select_node")


ToT nodes ready: generate_node, evaluate_branches_node, select_node


## TASK T3 — Wire Tree of Thoughts &nbsp;`6 pts`

Compile into **`tot_agent`**. Flow is linear: `START → generate → evaluate → select → END`. Node names: **`"generate"`**, **`"evaluate"`**, **`"select"`**.


In [ ]:
# TASK T3 — build and compile tot_agent
b = StateGraph(ToTState)
b.add_node("generate", generate_node)
b.add_node("evaluate", evaluate_branches_node)
b.add_node("select", select_node)
b.add_edge(START, "generate")
b.add_edge("generate", "evaluate")
b.add_edge("evaluate", "select")
b.add_edge("select", END)
tot_agent = b.compile()

<details><summary>💡 Hint</summary>

```python
b = StateGraph(ToTState)
b.add_node("generate", generate_node)
b.add_node("evaluate", evaluate_branches_node)
b.add_node("select", select_node)
b.add_edge(START, "generate")
b.add_edge("generate", "evaluate")
b.add_edge("evaluate", "select")
b.add_edge("select", END)
tot_agent = b.compile()
```
</details>

In [ ]:
# ✅ CHECK T3
try:
    nodes = set(tot_agent.get_graph().nodes.keys())
    ok = {"generate","evaluate","select"}.issubset(nodes)
    grader.check("T3", ok, 6, ok_msg="ToT wired with "+str(sorted(n for n in nodes if not n.startswith("__")))+".",
        fail_msg="Nodes found: "+str(sorted(nodes))+" (need 'generate','evaluate','select').")
except NameError:
    grader.check("T3", False, 6, fail_msg="`tot_agent` not defined.")
except Exception as e:
    grader.check("T3", False, 6, fail_msg="Error: "+str(e))


PASS  +6 pts  [Task T3]
   ToT wired with ['evaluate', 'generate', 'select'].

   100/120  [#########################-----]  83%
   Rank: 🌳 Branch Explorer - You search the space of ideas.



### ▶️ Run Tree of Thoughts live *(optional)*

In [ ]:
try:
    res = tot_agent.invoke({"problem": "Find a number under 50 that is a multiple of both 3 and 7.",
                            "thoughts": [], "scores": [], "best": "", "answer": ""})
    for t, s in zip(res["thoughts"], res["scores"]):
        print(f"[score {s}] {t}")
    print("\nChosen approach:", res["best"])
    print("Answer:", res["answer"])
except Exception as e:
    print("Live run needs your API key. Error:", e)


[score 8] List the multiples of 3 and 7 separately and find the common number under 50.
[score 9] Use the least common multiple (LCM) of 3 and 7 to directly find the smallest multiple under 50.
[score 8] Iterate through numbers under 50 and check each for divisibility by both 3 and 7.

Chosen approach: Use the least common multiple (LCM) of 3 and 7 to directly find the smallest multiple under 50.
Answer: ## Step 1: Find the least common multiple (LCM) of 3 and 7
To find the LCM of 3 and 7, we first list the multiples of each number. The multiples of 3 are 3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 33, 36, 39, 42, 45, 48. The multiples of 7 are 7, 14, 21, 28, 35, 42, 49. The smallest number that appears in both lists is 21, which is the LCM of 3 and 7.

## Step 2: Determine if the LCM is under 50
We compare the LCM (21) to the given limit (50). Since 21 is less than 50, it satisfies the condition of being under 50.

## Step 3: Verify if the LCM is a multiple of both 3 and 7
By definition, the

---
# ⚖️ Part 6 · Which Agent Wins Where?

You've built all five. The real skill is **choosing** the right one. Use what you know about each architecture's strengths and weaknesses.


## TASK CMP1 — Match the scenario to the best agent &nbsp;`10 pts`

Fill in **`scenario_best`** mapping each scenario to the agent that fits best. Use exactly these labels:
`"react"`, `"plan_execute"`, `"reflexion"`, `"codeact"`, `"tree_of_thoughts"`.

Scenarios:
- `"debug_unknown_error"` — you don't know the fix until you see the error
- `"structured_research_report"` — long task with clearly defined sections
- `"code_until_tests_pass"` — keep retrying until unit tests pass
- `"csv_data_analysis"` — compute month-over-month growth from a CSV
- `"game_of_24_puzzle"` — a puzzle with many possible paths to explore


In [ ]:
# TASK CMP1 — scenario_best = { ... }
scenario_best = {"debug_unknown_error": "react",
                 "structured_research_report": "plan_execute",
                 "code_until_tests_pass": "reflexion",
                 "csv_data_analysis": "codeact",
                 "game_of_24_puzzle": "tree_of_thoughts"}

<details><summary>💡 Hint</summary>

Think about each agent's defining trait:
- exploratory, "discover the problem as you go" → **react**
- everything plannable upfront → **plan_execute**
- clear pass/fail + retry with learning → **reflexion**
- computation / data manipulation → **codeact**
- large branching solution space → **tree_of_thoughts**
</details>

In [ ]:
# ✅ CHECK CMP1
key = {"debug_unknown_error":"react","structured_research_report":"plan_execute",
       "code_until_tests_pass":"reflexion","csv_data_analysis":"codeact",
       "game_of_24_puzzle":"tree_of_thoughts"}
try:
    got = {k: str(v).strip().lower() for k, v in scenario_best.items()}
    wrong = [k for k in key if got.get(k) != key[k]]
    grader.check("CMP1", len(wrong) == 0, 10,
        ok_msg="All five scenarios matched correctly.",
        fail_msg="Re-check these: "+str(wrong))
except NameError:
    grader.check("CMP1", False, 10, fail_msg="`scenario_best` not defined.")
except Exception as e:
    grader.check("CMP1", False, 10, fail_msg="Error: "+str(e))


PASS  +10 pts  [Task CMP1]
   All five scenarios matched correctly.

   110/120  [############################--]  92%
   Rank: 🌳 Branch Explorer - You search the space of ideas.



## TASK CMP2 — Match the defining trait to the agent &nbsp;`10 pts`

Fill in **`trait_best`** with the same five labels. Which agent…
- `"most_expensive_many_llm_calls"` — fires the most LLM calls (a call per branch)
- `"best_for_3_to_5_tool_calls"` — the sweet spot for short sequential tool use
- `"requires_a_secure_sandbox"` — must isolate execution for safety
- `"learns_from_failure_between_retries"` — writes a lesson after each failure
- `"decides_all_steps_before_acting"` — commits to the full plan upfront


In [ ]:
# TASK CMP2 — trait_best = { ... }
trait_best = {"best_for_3_to_5_tool_calls": "react",
              "decides_all_steps_before_acting": "plan_execute",
              "most_expensive_many_llm_calls":"tree_of_thoughts",
              "requires_a_secure_sandbox":"codeact",
              "learns_from_failure_between_retries":"reflexion"}

<details><summary>💡 Hint</summary>

Each line points straight at one architecture's headline property — re-read the intro of each Part if unsure.
</details>

In [ ]:
# ✅ CHECK CMP2
key2 = {"most_expensive_many_llm_calls":"tree_of_thoughts","best_for_3_to_5_tool_calls":"react",
        "requires_a_secure_sandbox":"codeact","learns_from_failure_between_retries":"reflexion",
        "decides_all_steps_before_acting":"plan_execute"}
try:
    got = {k: str(v).strip().lower() for k, v in trait_best.items()}
    wrong = [k for k in key2 if got.get(k) != key2[k]]
    grader.check("CMP2", len(wrong) == 0, 10,
        ok_msg="Every defining trait matched. You truly know your agents.",
        fail_msg="Re-check these: "+str(wrong))
except NameError:
    grader.check("CMP2", False, 10, fail_msg="`trait_best` not defined.")
except Exception as e:
    grader.check("CMP2", False, 10, fail_msg="Error: "+str(e))


PASS  +10 pts  [Task CMP2]
   Every defining trait matched. You truly know your agents.

   120/120  [##############################]  100%
   Rank: 🏛️ Agent Architect - You pick the right agent for the job.



---
## 🏆 Final Scorecard

In [ ]:
grader.scorecard()

  FINAL SCORECARD
  [x]  Task C1      6/6
  [x]  Task C2      8/8
  [x]  Task C3      6/6
  [x]  Task CMP1   10/10
  [x]  Task CMP2   10/10
  [x]  Task P1      6/6
  [x]  Task P2      8/8
  [x]  Task P3      6/6
  [x]  Task R1      6/6
  [x]  Task R2      8/8
  [x]  Task R3      6/6
  [x]  Task T1      6/6
  [x]  Task T2      8/8
  [x]  Task T3      6/6
  [x]  Task X1      6/6
  [x]  Task X2      8/8
  [x]  Task X3      6/6
  --------------------------------------------
  TOTAL  120/120   [##############################]  100%
  Rank: 🏛️ Agent Architect
  You pick the right agent for the job.


## 🎓 You built an agent zoo

Five architectures, each a different answer to *"how should an LLM act in the world?"*:
**ReAct** loops, **Plan-and-Execute** plans, **Reflexion** learns, **CodeAct** computes, **Tree of Thoughts** searches. There is no single best agent — only the best agent **for the task in front of you.**
